# 🕌 Athar - Embed Collections on Colab GPU

**Purpose:** Embed 7M+ documents from 10 collections using BAAI/bge-m3 on GPU

**Expected Time:** 13 hours (T4 free) or 3-4 hours (A100 paid)

**Output:** 10 Qdrant collections with full metadata

## Setup Instructions:
1. Runtime → Change runtime type → GPU (T4)
2. Run cells sequentially
3. Save embeddings to Google Drive periodically
4. Upload to Qdrant when complete

## Step 1: Verify GPU (1 minute)

In [ ]:
# Verify GPU is available
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# Expected output:
# GPU: Tesla T4
# GPU Memory: 15.8 GB

## Step 2: Install Dependencies (5 minutes)

In [ ]:
# Install required packages
!pip install -q transformers torch qdrant-client huggingface_hub tqdm

# Verify installations
import transformers
import qdrant_client
print(f"✅ Transformers: {transformers.__version__}")
print(f"✅ Qdrant Client: {qdrant_client.__version__}")

## Step 3: Mount Google Drive (Optional but RECOMMENDED)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create directories
import os
os.makedirs('/content/drive/MyDrive/Athar-Embeddings', exist_ok=True)
os.makedirs('/content/drive/MyDrive/Athar-Collections', exist_ok=True)

print("✅ Google Drive mounted")
print("📁 Created Athar directories")

## Step 4: Download Collections from HuggingFace (15-30 minutes)

In [ ]:
from huggingface_hub import snapshot_download

print("📥 Downloading collections from HuggingFace...")
print("This may take 15-30 minutes depending on collection sizes")

# Download all collections
try:\nexcept Exception as e:\n    print(f"⚠️  Download failed: {e}")\n    print("💡 Try Google Drive upload instead")\nsnapshot_download(
    repo_id="Kandil7/Athar-Datasets",
    local_dir="/content/athar-collections",
    repo_type="dataset"
)

# Verify download
!ls -lh /content/athar-collections/
!du -sh /content/athar-collections/

### Alternative: Upload from Google Drive

In [ ]:
# If you uploaded to Drive manually instead of HuggingFace
# Uncomment and run this cell instead

# !cp -r /content/drive/MyDrive/Athar-Collections /content/athar-collections
# !ls -lh /content/athar-collections/
# print("✅ Collections loaded from Google Drive")

## Step 5: Configure Settings (2 minutes)

In [ ]:
# Collection names (order by size: smallest first)
COLLECTIONS = [
    "seerah_passages",           # 0.8 GB
    "usul_fiqh",                 # 0.9 GB
    "spirituality_passages",     # 1.1 GB
    "aqeedah_passages",          # 1.8 GB
    "arabic_language_passages",  # 2.3 GB
    "quran_tafsir",              # 5.2 GB
    "islamic_history_passages",  # 6.0 GB
    "general_islamic",           # 6.5 GB
    "fiqh_passages",             # 7.0 GB
    "hadith_passages",           # 11.0 GB
]

# Paths
COLLECTIONS_DIR = "/content/athar-collections/collections"
EMBEDDINGS_DIR = "/content/drive/MyDrive/Athar-Embeddings"

# Qdrant configuration (update with your values)
QDRANT_URL = "http://your-qdrant-url:6333"
QDRANT_API_KEY = "your-api-key-here"

# Embedding settings
BATCH_SIZE = 512  # Optimal for T4 GPU (16GB VRAM)
MAX_LENGTH = 8192  # BGE-M3 max tokens

print(f"✅ Configuration set")
print(f"📚 {len(COLLECTIONS)} collections to process")
print(f"🎯 Batch size: {BATCH_SIZE}")

## Step 6: Load Embedding Model (3 minutes)

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch

print("🤖 Loading BAAI/bge-m3 model...")
print("This takes ~2 minutes to download and load")

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")
model = AutoModel.from_pretrained("BAAI/bge-m3")

# Move to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

print(f"✅ Model loaded on {device}")
print(f"📊 Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## Step 7: Embed All Collections (6-13 hours)

In [ ]:
import json
import time
import os
from tqdm import tqdm
import numpy as np
from datetime import datetime

# Progress tracking
PROGRESS_FILE = "/content/embedding_progress.json"

def log_progress(collection, status, elapsed, passage_count):
    """Log progress to file."""
    entry = {
        "collection": collection,
        "status": status,
        "passage_count": passage_count,
        "elapsed_minutes": elapsed / 60,
        "timestamp": datetime.now().isoformat()
    }
    
    with open(PROGRESS_FILE, 'a') as f:
        f.write(json.dumps(entry) + '\n')
    
    print(f"📝 Progress logged: {collection} - {status}")

def embed_collection(collection_name):
    """Embed a single collection."""
    filepath = f"{COLLECTIONS_DIR}/{collection_name}.jsonl"
    
    if not os.path.exists(filepath):
        print(f"⚠️  File not found: {filepath}")
        return None, None
    
    # Load passages
    print(f"\n📚 Loading {collection_name}...")
    passages = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                passages.append(json.loads(line))
            except:
                continue
    
    print(f"✅ Loaded {len(passages):,} passages")
    
    # Extract content
    contents = [p.get('content', '') for p in passages]
    
    # Embed in batches
    print(f"⚡ Embedding {len(contents):,} passages...")
    all_embeddings = []
    num_batches = (len(contents) + BATCH_SIZE - 1) // BATCH_SIZE
    
    start_time = time.time()
    
    for i in tqdm(range(0, len(contents), BATCH_SIZE), 
                  desc=f"Embedding {collection_name}",
                  unit="batch"):
        batch = contents[i:i + BATCH_SIZE]
        
        # Tokenize
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors='pt'
        ).to(device)
        
        # Embed
        with torch.no_grad():
            outputs = model(**encoded)
            # Use CLS token embedding
            embeddings = outputs.last_hidden_state[:, 0, :]
            # Normalize
            embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
        
        all_embeddings.extend(embeddings.cpu().tolist())
        
        # Progress report every 50 batches
        if (i // BATCH_SIZE) % 50 == 0 and i > 0:
            elapsed = time.time() - start_time
            batch_num = i // BATCH_SIZE
            speed = (i + BATCH_SIZE) / elapsed
            eta = (len(contents) - i - BATCH_SIZE) / speed
            print(f"  Batch {batch_num}/{num_batches} | "
                  f"Speed: {speed:.1f} batch/s | "
                  f"ETA: {eta/60:.1f} min")
    
    elapsed = time.time() - start_time
    
    print(f"\n✅ {collection_name}: {len(all_embeddings):,} embeddings in {elapsed/60:.1f} min")
    
    return passages, all_embeddings

# Process all collections
total_start = time.time()

for collection in COLLECTIONS:
    coll_start = time.time()
    
    print(f"\n{'='*70}")
    print(f"📖 Processing: {collection}")
    print(f"{'='*70}")
    
    log_progress(collection, "started", 0, 0)
    
    try:
        passages, embeddings = embed_collection(collection)
        
        if passages is None:
            log_progress(collection, "skipped", 0, 0)
            continue
        
        # Save embeddings to Drive (backup)
        np.save(f"{EMBEDDINGS_DIR}/{collection}_embeddings.npy", embeddings)
        
        # Save passages for later upload
        with open(f"{EMBEDDINGS_DIR}/{collection}_passages.json", 'w') as f:
            json.dump(passages, f)
        
        elapsed = time.time() - coll_start
        log_progress(collection, "completed", elapsed, len(passages))
        
        # Clear GPU memory
        torch.cuda.empty_cache()
        
        print(f"⏱️  Collection time: {elapsed/60:.1f} minutes")
        print(f"💾 Saved to Google Drive")
        
    except Exception as e:
        elapsed = time.time() - coll_start
        log_progress(collection, "failed", elapsed, 0)
        print(f"❌ Failed: {e}")
        import traceback
        traceback.print_exc()

total_elapsed = time.time() - total_start
print(f"\n{'='*70}")
print(f"✅ ALL COLLECTIONS EMBEDDED")
print(f"{'='*70}")
print(f"Total time: {total_elapsed/3600:.1f} hours")

## Step 8: Check Progress

In [ ]:
# View progress so far
!cat /content/embedding_progress.json | python -m json.tool

# List saved embeddings
!ls -lh /content/drive/MyDrive/Athar-Embeddings/

## Step 9: Upload to Qdrant (1-2 hours)

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# Connect to Qdrant
print("🔌 Connecting to Qdrant...")
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
print("✅ Connected")

def upload_collection_to_qdrant(collection_name):
    """Upload a collection to Qdrant."""
    
    # Load embeddings and passages
    embeddings = np.load(f"{EMBEDDINGS_DIR}/{collection_name}_embeddings.npy")
    with open(f"{EMBEDDINGS_DIR}/{collection_name}_passages.json", 'r') as f:
        passages = json.load(f)
    
    print(f"\n📤 Uploading {collection_name}...")
    print(f"  Passages: {len(passages):,}")
    print(f"  Embeddings: {len(embeddings):,}")
    
    # Create collection if not exists
    try:
        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=1024, distance=Distance.COSINE)
        )
        print(f"✅ Created collection: {collection_name}")
    except Exception as e:
        print(f"⚠️  Collection may exist: {e}")
    
    # Prepare points
    points = []
    for i, (passage, embedding) in enumerate(zip(passages, embeddings)):
        payload = {
            "content": passage.get("content", ""),
            "book_id": passage.get("book_id"),
            "title": passage.get("title", ""),
            "author": passage.get("author", ""),
            "author_death": passage.get("author_death"),
            "page": passage.get("page"),
            "chapter": passage.get("chapter", ""),
            "section": passage.get("section", ""),
            "category": passage.get("category", ""),
            "collection": collection_name,
        }
        
        points.append(PointStruct(
            id=i,
            vector=embedding.tolist(),
            payload=payload
        ))
    
    # Upload in batches
    batch_size = 2000
    for i in tqdm(range(0, len(points), batch_size), desc="Uploading"):
        batch = points[i:i + batch_size]
        client.upsert(
            collection_name=collection_name,
            points=batch
        )
    
    print(f"✅ Uploaded {len(points):,} vectors to {collection_name}")

# Upload all collections
for collection in COLLECTIONS:
    try:
        upload_collection_to_qdrant(collection)
    except Exception as e:
        print(f"❌ Failed to upload {collection}: {e}")

## Step 10: Verify Upload

In [ ]:
# Verify all collections
print("📊 Verifying collections...")
print()

total_vectors = 0

for collection in COLLECTIONS:
    try:
        info = client.get_collection(collection)
        print(f"📚 {collection}:")
        print(f"  Vectors: {info.points_count:,}")
        print(f"  Status: {info.status}")
        print(f"  Indexed: {info.indexed_vectors_count:,}")
        print()
        total_vectors += info.points_count
    except Exception as e:
        print(f"❌ {collection}: {e}")
        print()

print(f"{'='*70}")
print(f"✅ TOTAL VECTORS: {total_vectors:,}")
print(f"{'='*70}")

## Step 11: Test Retrieval

In [ ]:
def test_search(query, collection="fiqh_passages", top_k=5):
    """Test semantic search."""
    print(f"🔍 Query: {query}")
    print(f"📚 Collection: {collection}")
    print()
    
    # Embed query
    encoded = tokenizer(query, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model(**encoded)
        query_embedding = outputs.last_hidden_state[:, 0, :].cpu().tolist()[0]
    
    # Search
    results = client.search(
        collection_name=collection,
        query_vector=query_embedding,
        limit=top_k
    )
    
    print(f"✅ Found {len(results)} results\n")
    
    for i, result in enumerate(results):
        print(f"[{i+1}] Score: {result.score:.3f}")
        print(f"    Author: {result.payload.get('author')}")
        print(f"    Book: {result.payload.get('title')}")
        print(f"    Chapter: {result.payload.get('chapter')}")
        print(f"    Content: {result.payload['content'][:200]}...")
        print()

# Test queries
test_search("ما حكم صلاة الجماعة؟", "fiqh_passages")
test_search("حديث عن الصلاة", "hadith_passages")
test_search("التوحيد والإيمان", "aqeedah_passages")

## Step 12b: Backup Embeddings to HuggingFace (IMPORTANT!)

**Why backup:**
- Avoid re-embedding (saves 13+ hours)
- Share with team members
- Disaster recovery
- Free unlimited storage

**What gets backed up:**
- 10 embedding files (~7GB compressed)
- Passage metadata (~100MB)
- Total: ~8GB to HuggingFace

In [ ]:
# Install additional packages for backup
!pip install -q huggingface_hub
print("✅ Backup dependencies installed")

In [ ]:
import os
import gzip
import json
from huggingface_hub import HfApi, login
from datetime import datetime

# Configuration
HF_TOKEN = "hf_YOUR_TOKEN_HERE"  # Replace with your token
REPO_ID = "Kandil7/Athar-Embeddings"  # Your HF repo
EMBEDDINGS_DIR = "/content/drive/MyDrive/Athar-Embeddings"

# Login to HuggingFace
api = HfApi()
login(token=HF_TOKEN)

print(f"📦 BACKING UP EMBEDDINGS TO HUGGINGFACE")
print(f"   Repository: {REPO_ID}")
print(f"   Source: {EMBEDDINGS_DIR}")
print()

# Backup each collection
import glob
embedding_files = glob.glob(f"{EMBEDDINGS_DIR}/*_embeddings.npy")

for filepath in embedding_files:
    filename = os.path.basename(filepath)
    collection = filename.replace('_embeddings.npy', '')
    
    # Compress
    gz_path = f"/tmp/{collection}_embeddings.npy.gz"
    print(f"📦 Compressing {filename}...")
    
    with open(filepath, 'rb') as f_in:
        with gzip.open(gz_path, 'wb', compresslevel=6) as f_out:
            import shutil
            shutil.copyfileobj(f_in, f_out)
    
    original_size = os.path.getsize(filepath)
    compressed_size = os.path.getsize(gz_path)
    ratio = compressed_size / original_size
    
    print(f"   {original_size/1e9:.1f}GB → {compressed_size/1e9:.1f}GB ({ratio:.0%})")
    
    # Upload
    try:
        api.upload_file(
            path_or_fileobj=gz_path,
            path_in_repo=f"embeddings/{filename}.gz",
            repo_id=REPO_ID,
            repo_type="dataset"
        )
        print(f"   ✅ Uploaded")
    except Exception as e:
        print(f"   ❌ Failed: {e}")
    
    print()

print(f"{'='*70}")
print(f"✅ BACKUP COMPLETE")
print(f"{'='*70}")
print(f"🔗 View at: https://huggingface.co/datasets/{REPO_ID}")

## Step 12c: Restore Embeddings from HuggingFace (If Needed)

**When to use:**
- Colab disconnected and lost embeddings
- Want to download on different machine
- Team member needs embeddings

**This cell downloads and decompresses embeddings from HuggingFace**

In [ ]:
from huggingface_hub import hf_hub_download
import numpy as np

# Configuration
COLLECTIONS = [
    "seerah_passages",
    "usul_fiqh",
    "spirituality_passages",
    "aqeedah_passages",
    "arabic_language_passages",
    "quran_tafsir",
    "islamic_history_passages",
    "general_islamic",
    "fiqh_passages",
    "hadith_passages",
]

HF_TOKEN = "hf_YOUR_TOKEN_HERE"
REPO_ID = "Kandil7/Athar-Embeddings"
EMBEDDINGS_DIR = "/content/drive/MyDrive/Athar-Embeddings"

login(token=HF_TOKEN)

print(f"📥 RESTORING EMBEDDINGS FROM HUGGINGFACE")
print(f"   Repository: {REPO_ID}")
print()

import os
import glob

restored = 0
for collection in COLLECTIONS:
    filename = f"{collection}_embeddings.npy.gz"
    output_file = f"{EMBEDDINGS_DIR}/{collection}_embeddings.npy"
    
    # Check if already exists
    if os.path.exists(output_file):
        print(f"✅ {collection}: Already exists")
        restored += 1
        continue
    
    try:
        # Download
        print(f"📥 Downloading {collection}...")
        filepath = hf_hub_download(
            repo_id=REPO_ID,
            filename=f"embeddings/{filename}",
            repo_type="dataset"
        )
        
        # Decompress
        with gzip.open(filepath, 'rb') as f:
            embeddings = np.load(f)
        
        # Save
        np.save(output_file, embeddings)
        
        print(f"✅ {collection}: {len(embeddings):,} vectors restored")
        restored += 1
        
    except Exception as e:
        print(f"❌ {collection}: Failed - {e}")

print()
print(f"{'='*70}")
print(f"✅ RESTORE COMPLETE: {restored}/{len(COLLECTIONS)} collections")
print(f"{'='*70}")

## Step 12: Final Summary

In [ ]:
import json
from datetime import datetime

print(f"{'='*70}")
print(f"🕌 ATHAR - EMBEDDING PIPELINE COMPLETE")
print(f"{'='*70}")
print()

# Load progress
with open('/content/embedding_progress.json', 'r') as f:
    progress = [json.loads(line) for line in f]

completed = [p for p in progress if p['status'] == 'completed']
failed = [p for p in progress if p['status'] == 'failed']

print(f"✅ Completed: {len(completed)}/{len(COLLECTIONS)} collections")
print(f"❌ Failed: {len(failed)}/{len(COLLECTIONS)} collections")
print()

for p in completed:
    print(f"  ✅ {p['collection']}: {p['passage_count']:,} passages in {p['elapsed_minutes']:.1f} min")

if failed:
    print()
    for p in failed:
        print(f"  ❌ {p['collection']}: Failed")

total_time = sum(p['elapsed_minutes'] for p in completed)
total_passages = sum(p['passage_count'] for p in completed)

print()
print(f"{'='*70}")
print(f"📊 TOTALS:")
print(f"  Passages embedded: {total_passages:,}")
print(f"  Total time: {total_time/60:.1f} hours")
print(f"  Completed at: {datetime.now().isoformat()}")
print(f"{'='*70}")